In [6]:
import openmeteo_requests
import pandas as pd
import numpy as np
from retry_requests import retry

# 1. Setup the API client with retry logic
# If the connection drops, it will try 5 times before giving up
retry_session = retry(backoff_factor=0.2, retries=5)
openmeteo = openmeteo_requests.Client(session=retry_session)


params = {
    "latitude": 38.7167, #Lisboa
    "longitude": -9.1333, #Lisboa
    "start_date": "1940-01-01", 
    "end_date": "2025-01-01",
    "hourly": [
        "temperature_2m", "relative_humidity_2m", "surface_pressure", 
        "precipitation", "wind_speed_10m",
        "shortwave_radiation", "cloud_cover", "vapor_pressure_deficit", "wind_gusts_10m"
    ],
    "timezone": "UTC"
}

print("Fetching data from Open-Meteo API...")
responses = openmeteo.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)
response = responses[0]

# 3. Process the hourly data
hourly = response.Hourly()

# THE FIX: 'inclusive="left"' is the modern Pandas 2.0+ replacement for 'closed="left"'
date_range = pd.date_range(
    start=pd.to_datetime(hourly.Time(), unit="s"),
    end=pd.to_datetime(hourly.TimeEnd(), unit="s"),
    freq=pd.Timedelta(seconds=hourly.Interval()),
    inclusive="left"
)

# Organize raw arrays into a dictionary
weather_dict = {
    "date": date_range,
    "temperature": hourly.Variables(0).ValuesAsNumpy(),
    "humidity": hourly.Variables(1).ValuesAsNumpy(),
    "pressure": hourly.Variables(2).ValuesAsNumpy(),
    "precipitation": hourly.Variables(3).ValuesAsNumpy(),
    "wind_speed": hourly.Variables(4).ValuesAsNumpy(),
    "radiation": hourly.Variables(5).ValuesAsNumpy(),
    "cloud_cover": hourly.Variables(6).ValuesAsNumpy(),
    "vpd": hourly.Variables(7).ValuesAsNumpy(),
    "gusts": hourly.Variables(8).ValuesAsNumpy()
}

# 4. Create the DataFrame
df_hourly = pd.DataFrame(data=weather_dict)

# 5. Data Validation & Export
print(f"Successfully collected {len(df_hourly)} hours of data.")
print("-" * 30)
print(df_hourly.head()) # Shows the first 5 rows
print("-" * 30)

# Save the raw data to your 'data' folder
# This ensures we don't have to call the API every time we restart
df_hourly.to_csv("../data/weather_raw.csv", index=False)
print("Data saved to data/weather_raw.csv")

Fetching data from Open-Meteo API...
Successfully collected 745152 hours of data.
------------------------------
                 date  temperature   humidity    pressure  precipitation  \
0 1940-01-01 00:00:00      11.4735  89.000771  993.264038            NaN   
1 1940-01-01 01:00:00      11.9735  90.539131  992.975769            NaN   
2 1940-01-01 02:00:00      12.2235  90.858429  992.483582            NaN   
3 1940-01-01 03:00:00      12.4235  90.570908  992.984863            NaN   
4 1940-01-01 04:00:00      12.4235  91.781021  993.084229            NaN   

   wind_speed  radiation  cloud_cover       vpd  gusts  
0   37.750252        NaN        100.0  0.149212    NaN  
1   35.583591        NaN        100.0  0.132642    NaN  
2   34.726475        NaN         95.0  0.130288    NaN  
3   33.557331        NaN         99.0  0.136161    NaN  
4   31.421213        NaN         99.0  0.118685    NaN  
------------------------------
Data saved to data/weather_raw.csv
